# Tagalog ⇄ Cuyonon — train on Colab

QLoRA fine-tune of `Qwen/Qwen2.5-1.5B-Instruct`, both directions, on a free **T4**.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.
New Colab notebooks default to a CPU runtime and nothing here will work on CPU.

Run the cells top to bottom. No kernel restart is needed — training/eval run as
subprocesses, so they pick up freshly installed packages.

## 1. Confirm the GPU

In [ ]:
import subprocess, torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → T4 GPU, then re-run this cell."
)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

## 2. Get the code

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("translator"):
    !git clone https://github.com/jubilcabrestante/translator.git
os.chdir("/content/translator")
!git pull --ff-only
print("cwd:", os.getcwd())

## 3. Install dependencies

`requirements.txt` is the Windows-tested set. `bitsandbytes` is then upgraded
because the pinned 0.44.1 has no CUDA-12.8 binary and imports a Triton API that
current Colab removed. Pip will print dependency-conflict warnings about
`gradio` / `google-adk` / `gcsfs` — those packages are unrelated to training and
the warnings are safe to ignore.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes
import bitsandbytes, importlib.metadata as md
print("bitsandbytes", md.version("bitsandbytes"), "OK")

## 4. Where to write checkpoints

Colab wipes the VM disk on disconnect (idle ~90 min, hard cap ~12 h). Mount Drive
so a checkpoint survives and the training cell can resume. Skip the mount and
`OUT_DIR` stays on the ephemeral disk — fine for a single uninterrupted run.

In [ ]:
OUT_DIR = "outputs/translator-lora"  # ephemeral VM disk

# --- To persist to Drive instead, uncomment these three lines: ---
# from google.colab import drive
# drive.mount("/content/drive")
# OUT_DIR = "/content/drive/MyDrive/translator/outputs/translator-lora"

print("checkpoints →", OUT_DIR)

## 5. Build the dataset

Reads `assets/taga-cuyo.txt` + `assets/entities/`, writes `data/train.jsonl` and
`data/val.jsonl` (both directions + proper-noun passthrough).

In [ ]:
!python scripts/prepare_data.py

## 6. Train

~3 epochs on the ~4.7k examples — a few minutes on a T4. On a bigger GPU add
`--model Qwen/Qwen2.5-7B-Instruct --batch-size 4`. If `OUT_DIR` already holds a
checkpoint (e.g. after a disconnect) this resumes from the latest one.

In [ ]:
import glob
cks = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"),
             key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = f"--resume-from-checkpoint {cks[-1]}" if cks else ""
print("resume:", resume or "(fresh run)")
!python scripts/train.py --output-dir {OUT_DIR} {resume}

## 7. Evaluate (chrF / BLEU / exact-match)

In [ ]:
!python scripts/evaluate.py --adapter-dir {OUT_DIR} --limit 200

## 8. Spot-check a few translations

In [ ]:
!python scripts/translate.py --adapter-dir {OUT_DIR} --text "Magandang umaga sa inyong lahat." --to Cuyonon
!python scripts/translate.py --adapter-dir {OUT_DIR} --text "Saan ka pupunta?" --to Cuyonon

## 9. Save the adapter off the VM

If you mounted Drive in step 4 it is already saved there. Otherwise zip just the
adapter (not the multi-hundred-MB optimizer checkpoints) and download it.

In [ ]:
!cd "{OUT_DIR}" && zip -qr /content/translator-lora.zip . -x 'checkpoint-*/*'
from google.colab import files
files.download("/content/translator-lora.zip")